### ERA5 1° monthly and daily means from public hourly data (ARCO-ERA5)

A data-preparation step for E1–E5: builds the ERA5 daily reference (E4) and ERA5 monthly means on the regular 1° grid of the evaluations, from public data only. The monthly means served as the evaluations' monthly reference until DKRZ published its conservatively regridded ESMValTool files (`ERA5/mon_1deg_dkrz_v3`), which the evaluations now use; they agree with them closely (`tas` to 0.001 K RMS) and remain an independent check.

- **Source:** hourly ERA5 reanalysis at 0.25° from the public ARCO-ERA5 raw mirror, `gs://gcp-public-data-arco-era5/raw` (Google's copy of the Copernicus Climate Change Service data, one NetCDF file per day, variable and level). It is read anonymously; no credentials are needed.
- **Method:** each daily mean is the mean of the day's 24 hourly values (00–23 UTC), and each monthly mean is the mean of all hourly values in the month, both computed at 0.25°. The means are then regridded conservatively onto the regular 1° grid (180 × 360, cell centres at half degrees) with xESMF. This is the same method the evaluations use to regrid the models; the previous `ERA5/mon_1deg` files were point-interpolated, and the previous daily file averaged four 6-hourly values on another grid.
- **Variables:** `tas`, `ts` (skin temperature everywhere), `ts_spec` (`ts` as the AIMIP protocol defines it: skin temperature over land and sea ice, SST over open ocean), `ps`, `psl`, `uas`, `vas`, `tdas`, `huss`, `pr`, and `ta`, `hus`, `ua`, `va` at 1000, 850, 700, 500, 250, 100 and 50 hPa, plus `zg` at 500 hPa.
  - `huss` is computed from hourly 2 m dewpoint and surface pressure before averaging, so it is not affected by the nonlinearity of converting a mean dewpoint. `tdas` is the mean dewpoint; with `ps` it gives `huss` the way the evaluations convert submissions that provide dewpoint (`convert_tdas_to_huss`), so both conversions are available.
  - `ts_spec` is blended hour by hour at 0.25°: where ERA5 has an SST (the ocean), the sea-ice fraction takes the skin temperature and the open water the SST; elsewhere it is the skin temperature. `ts` keeps the convention of the previous `mon_1deg` files (skin temperature everywhere), so either can serve as the reference.
  - `pr` comes from the hourly accumulated total precipitation. Each hourly value covers the hour ending at its time stamp, so a day runs from 01 UTC to 00 UTC of the next day, and a month from 01 UTC on its first day to 00 UTC on the first day of the next month. It differs by about 2% RMS (no global-mean difference) from ERA5's separately published monthly-mean precipitation, which is built from the forecast accumulations differently; the monthly means of instantaneous fields such as `tas` match that product to 0.001 K.
- **Period:** monthly means January 1978 to December 2024; daily means for the two periods of the AIMIP daily output, 1 October 1978 to 31 December 1979 and the year 2024. The daily means come from the same reads as the monthly means of those months.
- **Output:** monthly: `ERA5/mon_1deg_arco/`, one file per variable, with the same file names as `ERA5/mon_1deg`, so the evaluations switch reference by directory alone. Daily: `ERA5/day_1deg_arco/ERA5_daily_1deg_19781001-19791231.nc` and `…_20240101-20241231.nc`, one file per period in the layout of `ERA5/day_1deg` (dimensions `time`, `latitude`, `longitude`, `plev`; time labelled at 00 UTC of each day), plus `tdas`.

**Cost.** The run reads about 32 TB from the public bucket (roughly 50 MB per day, variable and level). On an n2-highmem-16 in the bucket's region (us-central1) with 16 worker processes it takes about 36 s per month, so about 6 hours in all, with up to about 55 GB of memory in use (more for the months with daily output); set `ARCO_PROCESSES` for fewer workers and less memory. Each month is written to a small checkpoint file as soon as it is done, so an interrupted run resumes where it stopped. Set the environment variable `ONLY_MONTHS` (e.g. `1979-01`) to run a few months first, for example to time a single month.

Paths are relative to `AIMIP_DATA_ROOT`, as in the evaluation notebooks; the default, `../../../local_data`, is the repo's `local_data/` as seen from this directory. Output files are written to a temporary file and renamed into place, so hard links to an existing file keep the old contents.

In [1]:
import concurrent.futures as cf
import multiprocessing as mp
import os
import time
from datetime import datetime, timezone

import fsspec
import io
import numpy as np
import pandas as pd
import xarray as xr
import xesmf

import arco_era5
from arco_era5 import SURFACE, PRESSURE, month_mean

In [2]:
PERIOD = ('1978-01', '2024-12')
DATA_ROOT = os.environ.get('AIMIP_DATA_ROOT', '../../../local_data')
OUTPUT_DIR = os.path.join(DATA_ROOT, 'ERA5/mon_1deg_arco')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
FILE_TEMPLATE = 'native6_ERA5_an_v1_Amon_{var}_1978-2024.nc'  # same names as ERA5/mon_1deg
DAILY_PERIODS = [('1978-10-01', '1979-12-31'), ('2024-01-01', '2024-12-31')]  # whole months: the AIMIP daily output periods
DAILY_OUTPUT = os.path.join(DATA_ROOT, 'ERA5/day_1deg_arco/ERA5_daily_1deg_{}-{}.nc')  # first period: same name as ERA5/day_1deg
DAILY_CHECKPOINT_DIR = os.path.join(CHECKPOINT_DIR, 'daily')
N_PROCESSES = int(os.environ.get('ARCO_PROCESSES', '16'))  # worker processes, each reading one variable-level-month at a time
MONTHS_IN_FLIGHT = 2  # months queued at once, so workers stay busy across month boundaries
ONLY_MONTHS = os.environ.get('ONLY_MONTHS')  # e.g. '1979-01,1979-02' to process only those months
ONLY_MONTHS = ONLY_MONTHS.split(',') if ONLY_MONTHS else None

os.makedirs(DAILY_CHECKPOINT_DIR, exist_ok=True)
months = [m for m in pd.period_range(*PERIOD, freq='M') if ONLY_MONTHS is None or str(m) in ONLY_MONTHS]
daily_months = set().union(*(pd.period_range(*period, freq='M') for period in DAILY_PERIODS))
print(f'{len(months)} months: {months[0]} to {months[-1]}; {len(daily_months & set(months))} of them with daily means')

564 months: 1978-01 to 2024-12; 27 of them with daily means


Variables, the same for the monthly and daily output. Each output variable is a mean of one hourly ARCO field (`plain`), or of a quantity derived hourly from several (`huss` from dewpoint and surface pressure, `ts_spec` from skin temperature, SST and sea-ice cover, `zg` from geopotential, `pr` from accumulated precipitation). Metadata follows the CMIP conventions of the previous `mon_1deg` files.

In [3]:
PRESSURE_LEVELS_HPA = [1000, 850, 700, 500, 250, 100, 50]

# CMIP name: (ARCO tree, ARCO variable, levels in hPa or None, kind)
VARIABLES = {
    'tas': (SURFACE, '2m_temperature', None, 'plain'),
    'ts': (SURFACE, 'skin_temperature', None, 'plain'),
    'ts_spec': (SURFACE, ('skin_temperature', 'sea_surface_temperature', 'sea_ice_cover'), None, 'ts_spec'),
    'ps': (SURFACE, 'surface_pressure', None, 'plain'),
    'psl': (SURFACE, 'mean_sea_level_pressure', None, 'plain'),
    'uas': (SURFACE, '10m_u_component_of_wind', None, 'plain'),
    'vas': (SURFACE, '10m_v_component_of_wind', None, 'plain'),
    'tdas': (SURFACE, '2m_dewpoint_temperature', None, 'plain'),
    'huss': (SURFACE, ('2m_dewpoint_temperature', 'surface_pressure'), None, 'huss'),
    'pr': (SURFACE, 'total_precipitation', None, 'pr'),
    'ta': (PRESSURE, 'temperature', PRESSURE_LEVELS_HPA, 'plain'),
    'hus': (PRESSURE, 'specific_humidity', PRESSURE_LEVELS_HPA, 'plain'),
    'ua': (PRESSURE, 'u_component_of_wind', PRESSURE_LEVELS_HPA, 'plain'),
    'va': (PRESSURE, 'v_component_of_wind', PRESSURE_LEVELS_HPA, 'plain'),
    'zg': (PRESSURE, 'geopotential', [500], 'zg'),
}
METADATA = {  # standard_name, long_name, units
    'tas': ('air_temperature', 'Near-Surface Air Temperature', 'K'),
    'ts': ('surface_temperature', 'Surface Temperature', 'K'),
    'ts_spec': ('surface_temperature', 'Surface Temperature (skin temperature over land and sea ice, SST over open ocean)', 'K'),
    'ps': ('surface_air_pressure', 'Surface Air Pressure', 'Pa'),
    'psl': ('air_pressure_at_mean_sea_level', 'Sea Level Pressure', 'Pa'),
    'uas': ('eastward_wind', 'Eastward Near-Surface Wind', 'm s-1'),
    'vas': ('northward_wind', 'Northward Near-Surface Wind', 'm s-1'),
    'tdas': ('dew_point_temperature', '2m Dewpoint Temperature', 'K'),
    'huss': ('specific_humidity', 'Near-Surface Specific Humidity', '1'),
    'pr': ('precipitation_flux', 'Precipitation', 'kg m-2 s-1'),
    'ta': ('air_temperature', 'Air Temperature', 'K'),
    'hus': ('specific_humidity', 'Specific Humidity', '1'),
    'ua': ('eastward_wind', 'Eastward Wind', 'm s-1'),
    'va': ('northward_wind', 'Northward Wind', 'm s-1'),
    'zg': ('geopotential_height', 'Geopotential Height', 'm'),
}

Reading. The worker functions are in `arco_era5.py`, so that a process pool can import them: each task fetches one month of one variable at one level in a single batched request and returns its 0.25° mean. Raw files hold one day's 24 hourly fields packed as 16-bit integers; since the packing is linear, a month's sum is taken on the integers and unpacked once per file. `huss` is the exception: it is converted hour by hour. For `pr`, each hourly accumulation covers the hour ending at its time stamp, so a month uses hours 01–23 of each day plus 00 UTC of the following day.

In [4]:
def read_grid():
    """Latitude and longitude of the 0.25-degree source grid."""
    fs = fsspec.filesystem('gs', token='anon')
    raw = fs.cat(arco_era5.raw_path(SURFACE, '2m_temperature', pd.Timestamp('1979-01-01')))
    with xr.open_dataset(io.BytesIO(raw), engine='scipy') as ds:
        return ds.latitude.values.astype(np.float64), ds.longitude.values.astype(np.float64)


def month_days(month):
    return list(pd.date_range(month.start_time.normalize(), month.end_time.normalize(), freq='D'))


def month_tasks(month, variables):
    """{(var, level): month_mean arguments} for one month."""
    days = month_days(month)
    tasks = {}
    for var in variables:
        tree, variable, levels, kind = VARIABLES[var]
        for level in levels or [None]:
            tasks[(var, level)] = (tree, variable, days, level, kind, days[-1] + pd.Timedelta(days=1), month in daily_months)
    return tasks

Grids and the conservative regridder. The target is the regular 1° grid of the evaluations, defined here rather than read from another file. The source cells are 0.25° wide, with the polar rows clipped at ±90°.

In [5]:
src_lat, src_lon = read_grid()
order = np.argsort(src_lat)  # ascending latitude for the regridder
src_lat = src_lat[order]


def edges(centres, step, lo=None, hi=None):
    e = np.append(centres - step / 2, centres[-1] + step / 2)
    return np.clip(e, lo, hi) if lo is not None else e


source_grid = xr.Dataset(coords={
    'lat': ('lat', src_lat), 'lon': ('lon', src_lon),
    'lat_b': ('lat_b', edges(src_lat, 0.25, -90, 90)), 'lon_b': ('lon_b', edges(src_lon, 0.25)),
})
target_lat, target_lon = np.arange(-89.5, 90), np.arange(0.5, 360)
target_grid = xr.Dataset(coords={
    'lat': ('lat', target_lat), 'lon': ('lon', target_lon),
    'lat_b': ('lat_b', np.arange(-90.0, 91)), 'lon_b': ('lon_b', np.arange(0.0, 361)),
})
regridder = xesmf.Regridder(source_grid, target_grid, 'conservative', periodic=True)


def to_1deg(field):
    """Regrid (..., lat, lon) at 0.25 degrees, latitude from 90 to -90, to the 1-degree grid."""
    dims = tuple(f'd{i}' for i in range(field.ndim - 2)) + ('lat', 'lon')
    return regridder(xr.DataArray(field[..., order, :], dims=dims, coords={'lat': src_lat, 'lon': src_lon})).values

Monthly checkpoints. Each month's 1° fields go to `checkpoints/{var}_{YYYYMM}.nc`, and in the daily period its daily fields to `checkpoints/daily/{var}_{YYYYMM}.nc`; months already checkpointed are skipped, so rerunning this cell resumes an interrupted run.

In [6]:
def checkpoint_path(var, month, daily=False):
    return os.path.join(DAILY_CHECKPOINT_DIR if daily else CHECKPOINT_DIR, f'{var}_{month.strftime("%Y%m")}.nc')


def write_checkpoint(var, month, fields, times, daily=False):
    """fields: {level: 0.25-degree array (time, lat, lon)}; written as (time, [plev,] lat, lon) at 1 degree."""
    levels = VARIABLES[var][2]
    if levels is None:
        da = xr.DataArray(to_1deg(fields[None]), dims=('time', 'lat', 'lon'))
    else:
        data = np.stack([to_1deg(fields[level]) for level in levels], axis=1)
        da = xr.DataArray(data, dims=('time', 'plev', 'lat', 'lon'), coords={'plev': np.array(levels, float) * 100})
    da = da.assign_coords(time=times, lat=target_lat, lon=target_lon).astype(np.float32)
    path = checkpoint_path(var, month, daily)
    da.to_dataset(name=var).to_netcdf(path + '.tmp')
    os.replace(path + '.tmp', path)


def finish(month, futures):
    results = {key: future.result() for key, future in futures.items()}
    daily = month in daily_months
    for var in sorted({var for var, _ in results}):
        own = {level: r for (v, level), r in results.items() if v == var}
        if daily:
            write_checkpoint(var, month, {level: d for level, (_, d) in own.items()}, month_days(month), daily=True)
            own = {level: m for level, (m, _) in own.items()}
        write_checkpoint(var, month, {level: m[None] for level, m in own.items()}, [month.start_time])


def missing(var, month):
    return not os.path.exists(checkpoint_path(var, month)) or (
        month in daily_months and not os.path.exists(checkpoint_path(var, month, daily=True)))


t_start = time.time()
todo = {m: [v for v in VARIABLES if missing(v, m)] for m in months}
todo = {m: v for m, v in todo.items() if v}
print(f'{len(todo)} of {len(months)} months to do', flush=True)
in_flight = {}  # month -> {(var, level): future}
with cf.ProcessPoolExecutor(N_PROCESSES, mp_context=mp.get_context('spawn')) as pool:
    for i, (month, variables) in enumerate(todo.items()):
        in_flight[month] = {key: pool.submit(month_mean, *args) for key, args in month_tasks(month, variables).items()}
        while len(in_flight) >= MONTHS_IN_FLIGHT or (i == len(todo) - 1 and in_flight):
            done_month = next(iter(in_flight))
            finish(done_month, in_flight.pop(done_month))
            hours = (time.time() - t_start) / 3600
            print(f'{done_month} done; {hours:.2f} h so far', flush=True)

12 of 564 months to do


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


...

/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


/home/brianhenn/miniconda3/envs/aimip-evaluations/lib/python3.11/site-packages/xesmf/smm.py:131: UserWarning: Input array is not C_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not C_CONTIGUOUS. ' 'Will affect performance.')


2024-12 done; 0.24 h so far


Assemble one file per variable from the monthly checkpoints, with time stamped mid-month and `time_bnds` as in the previous `mon_1deg` files. Requires every month of `PERIOD`; skipped when `ONLY_MONTHS` is set.

In [7]:
def assemble(var):
    all_months = pd.period_range(*PERIOD, freq='M')
    missing = [str(m) for m in all_months if not os.path.exists(checkpoint_path(var, m))]
    assert not missing, f'{var}: {len(missing)} months missing, e.g. {missing[:3]}'
    da = xr.concat([xr.open_dataset(checkpoint_path(var, m))[var].load() for m in all_months], dim='time')
    starts = all_months.start_time
    ends = (all_months + 1).start_time
    da = da.assign_coords(time=starts + (ends - starts) / 2)
    standard_name, long_name, units = METADATA[var]
    ds = da.assign_attrs(standard_name=standard_name, long_name=long_name, units=units).to_dataset(name=var)
    ds['time_bnds'] = xr.DataArray(np.stack([starts, ends], axis=1), dims=('time', 'bnds'))
    ds['lat_bnds'] = xr.DataArray(np.stack([target_lat - 0.5, target_lat + 0.5], axis=1), dims=('lat', 'bnds'))
    ds['lon_bnds'] = xr.DataArray(np.stack([target_lon - 0.5, target_lon + 0.5], axis=1), dims=('lon', 'bnds'))
    ds.time.attrs.update(bounds='time_bnds', standard_name='time', axis='T')
    ds.lat.attrs.update(bounds='lat_bnds', standard_name='latitude', units='degrees_north', axis='Y')
    ds.lon.attrs.update(bounds='lon_bnds', standard_name='longitude', units='degrees_east', axis='X')
    if 'plev' in ds.dims:
        ds.plev.attrs.update(standard_name='air_pressure', long_name='pressure', units='Pa', axis='Z', positive='down')
    ds.attrs = {
        'Conventions': 'CF-1.7',
        'title': f'ERA5 monthly mean {var} on a regular 1-degree grid',
        'source': 'ERA5 hourly reanalysis (Copernicus Climate Change Service), read from the public ARCO-ERA5 '
                  f'raw mirror {arco_era5.ARCO_RAW}',
        'method': 'Monthly mean of all hourly values at 0.25 degrees (huss computed hourly from 2 m dewpoint and '
                  'surface pressure; ts_spec blended hourly from skin temperature, SST and sea-ice cover; pr from hourly '
                  'accumulated total precipitation), conservatively regridded '
                  'to the regular 1-degree grid with xESMF',
        'history': f'{datetime.now(timezone.utc):%Y-%m-%d} created by '
                   'evaluations/notebooks/data_prep/ERA5-monthly-1deg-ARCO.ipynb (https://github.com/ai2cm/AIMIP)',
    }
    encoding = {'time': {'units': 'days since 1850-1-1 00:00:00', 'calendar': 'standard'},
                'time_bnds': {'units': 'days since 1850-1-1 00:00:00', 'calendar': 'standard'}}
    path = os.path.join(OUTPUT_DIR, FILE_TEMPLATE.format(var=var))
    tmp = path + '.tmp'
    ds.to_netcdf(tmp, encoding=encoding)
    os.replace(tmp, path)
    return path


if ONLY_MONTHS is None:
    for var in VARIABLES:
        print(assemble(var), flush=True)
else:
    print('ONLY_MONTHS is set: not assembling the full-period files')

/tmp/ipykernel_156437/1820792416.py:35: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1850-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  ds.to_netcdf(tmp, encoding=encoding)


/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_tas_1978-2024.nc


/tmp/ipykernel_156437/1820792416.py:35: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1850-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  ds.to_netcdf(tmp, encoding=encoding)


/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_ts_1978-2024.nc


/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_ts_spec_1978-2024.nc


...

/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_ua_1978-2024.nc


/tmp/ipykernel_156437/1820792416.py:35: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1850-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  ds.to_netcdf(tmp, encoding=encoding)


/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_va_1978-2024.nc


/tmp/ipykernel_156437/1820792416.py:35: UserWarning: Times can't be serialized faithfully to int64 with requested units 'days since 1850-01-01'. Resolution of 'hours' needed. Serializing times to floating point instead. Set encoding['dtype'] to integer dtype to serialize to int64. Set encoding['dtype'] to floating point dtype to silence this warning.
  ds.to_netcdf(tmp, encoding=encoding)


/home/brianhenn/dev/AIMIP/local_data/ERA5/mon_1deg_arco/native6_ERA5_an_v1_Amon_zg_1978-2024.nc


Assemble one daily file per period from the daily checkpoints, in the layout of the previous `ERA5/day_1deg` file: dimensions `time` (00 UTC of each day), `latitude`, `longitude` and `plev`, pressure-level variables as (`plev`, `time`, `latitude`, `longitude`), and `zg` at 500 hPa as a single-level field. Requires every month of `DAILY_PERIODS`; skipped when `ONLY_MONTHS` is set.

In [8]:
def assemble_daily(period):
    all_months = pd.period_range(*period, freq='M')
    ds = xr.Dataset()
    for var in VARIABLES:
        lost = [str(m) for m in all_months if not os.path.exists(checkpoint_path(var, m, daily=True))]
        assert not lost, f'{var}: {len(lost)} months of daily means missing, e.g. {lost[:3]}'
        da = xr.concat([xr.open_dataset(checkpoint_path(var, m, daily=True))[var].load() for m in all_months], dim='time')
        if var == 'zg':
            da = da.squeeze('plev', drop=True)
        elif 'plev' in da.dims:
            da = da.transpose('plev', 'time', 'lat', 'lon')
        standard_name, long_name, units = METADATA[var]
        if var == 'zg':
            long_name = 'Geopotential Height at 500 hPa'
        ds[var] = da.assign_attrs(standard_name=standard_name, long_name=long_name, units=units)
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})
    ds.latitude.attrs.update(standard_name='latitude', units='degrees_north', axis='Y')
    ds.longitude.attrs.update(standard_name='longitude', units='degrees_east', axis='X')
    ds.plev.attrs.update(standard_name='air_pressure', long_name='pressure', units='Pa', axis='Z', positive='down')
    ds.attrs = {
        'Conventions': 'CF-1.7',
        'title': f'ERA5 daily means on a regular 1-degree grid, {period[0]} to {period[1]}',
        'source': 'ERA5 hourly reanalysis (Copernicus Climate Change Service), read from the public ARCO-ERA5 '
                  f'raw mirror {arco_era5.ARCO_RAW}',
        'method': 'Daily mean of the 24 hourly values (00-23 UTC) at 0.25 degrees, labelled with 00 UTC of the day '
                  '(huss computed hourly from 2 m dewpoint and surface pressure; pr from hourly accumulated total '
                  'precipitation, 01 UTC to 00 UTC of the next day; ts is skin temperature everywhere; ts_spec is skin '
                  'temperature over land and sea ice and SST over open ocean, blended hourly), '
                  'conservatively regridded to the regular 1-degree grid with xESMF',
        'history': f'{datetime.now(timezone.utc):%Y-%m-%d} created by '
                   'evaluations/notebooks/data_prep/ERA5-monthly-1deg-ARCO.ipynb (https://github.com/ai2cm/AIMIP)',
    }
    path = DAILY_OUTPUT.format(*(p.replace('-', '') for p in period))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ds.to_netcdf(path + '.tmp', encoding={'time': {'units': 'hours since 1940-01-01', 'calendar': 'standard'}})
    os.replace(path + '.tmp', path)
    return path


if ONLY_MONTHS is None:
    for period in DAILY_PERIODS:
        print(assemble_daily(period), flush=True)
else:
    print('ONLY_MONTHS is set: not assembling the daily file')

/home/brianhenn/dev/AIMIP/local_data/ERA5/day_1deg_arco/ERA5_daily_1deg_19781001-19791231.nc


/home/brianhenn/dev/AIMIP/local_data/ERA5/day_1deg_arco/ERA5_daily_1deg_20240101-20241231.nc


Optional check against the previous files. This does not feed the output; it only runs if the previous `ERA5/mon_1deg` files are present. Differences are expected: the previous files are point-interpolated (about 0.3 K RMS for `tas`), and their `huss` (here from `tdas`) is converted after averaging.

In [9]:
previous_dir = os.path.join(DATA_ROOT, 'ERA5/mon_1deg')
check_month = months[0]
if os.path.isdir(previous_dir):
    w = np.cos(np.deg2rad(target_lat))[:, None] * np.ones((1, len(target_lon)))
    for var in ['tas', 'ts', 'ps', 'psl', 'uas', 'tdas', 'pr', 'ta', 'zg']:
        new = xr.open_dataset(checkpoint_path(var, check_month))[var].isel(time=0)
        old_path = os.path.join(previous_dir, FILE_TEMPLATE.format(var=var))
        old = xr.open_dataset(old_path, decode_timedelta=False)[var]
        old = old.sel(time=old.time.dt.strftime('%Y-%m') == str(check_month)).isel(time=0)
        if 'plev' in new.dims:
            new, old = new.sel(plev=50000), old.sel(plev=50000)
        d = (new.values - old.values)
        ok = np.isfinite(d)
        print(f'{var:5s} {check_month}: global-mean new {np.average(new.values[ok], weights=w[ok]):.6g}, '
              f'previous {np.average(old.values[ok], weights=w[ok]):.6g}, RMS difference {np.sqrt(np.average(d[ok] ** 2, weights=w[ok])):.3g}')

tas   1978-01: global-mean new 285.036, previous 285.036, RMS difference 0.357
ts    1978-01: global-mean new 285.734, previous 285.734, RMS difference 0.519
ps    1978-01: global-mean new 98540.1, previous 98541.3, RMS difference 554


psl   1978-01: global-mean new 101151, previous 101151, RMS difference 14.8
uas   1978-01: global-mean new -0.28734, previous -0.288159, RMS difference 0.154


tdas  1978-01: global-mean new 280.535, previous 280.535, RMS difference 0.331
pr    1978-01: global-mean new 3.31858e-05, previous 3.31851e-05, RMS difference 2.16e-09
ta    1978-01: global-mean new 257.343, previous 257.343, RMS difference 0.0177
zg    1978-01: global-mean new 5630.36, previous 5630.36, RMS difference 0.378
